In [1]:
import pandas as pd
import os
import subprocess
from collections import Counter

INPUT_FILE = "train_paraphrased.csv"
CLEANED_FILE = "data_new/train_paraphrased_clean.csv"

In [3]:
print("加载 train_paraphrased.csv ...")
df = pd.read_csv(INPUT_FILE)

print(f"原始行数: {len(df):,}")

df.head(20)

加载 train_paraphrased.csv ...
原始行数: 47,964


,user_query,dialogue_context,retrieved_docs,true_q_i,true_a_i,dataset,split,original_id,paraphrase_rank
0,How does T-cell count contribute to the severi...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,train,358_p1,1
1,What significance does T-cell count hold in th...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,train,358_p2,2
2,Can elevated or reduced T-cell counts serve as...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,train,358_p3,3
3,In what way is T-cell count linked to the deve...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,train,358_p4,4
4,Does the relationship between T-cell count and...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,train,358_p5,5
5,Is there evidence to suggest that individuals ...,NaN,"[""Title: Emergent severe acute respiratory dis...",What role does T-cell count play in severe hum...,The T-cell count plays a crucial role in sever...,covidqa,train,358_p6,6
6,What does MVO stand for?,NaN,"[""Title: Gene Knockdowns in Adult Animals: PPM...",What is MVO?,MVO is not defined or mentioned in the provide...,covidqa,train,457_p1,1
7,Is MVO an abbreviation of some sort?,NaN,"[""Title: Gene Knockdowns in Adult Animals: PPM...",What is MVO?,MVO is not defined or mentioned in the provide...,covidqa,train,457_p2,2
8,Can you define what MVO means?,NaN,"[""Title: Gene Knockdowns in Adult Animals: PPM...",What is MVO?,MVO is not defined or mentioned in the provide...,covidqa,train,457_p3,3
9,Do you know the definition of MVO?,NaN,"[""Title: Gene Knockdowns in Adult Animals: PPM...",What is MVO?,MVO is not defined or mentioned in the provide...,covidqa,train,457_p4,4


In [12]:
# add_question_id_to_paraphrased.py
# 给 train_paraphrased.csv 加上 Question_ID（原始问题的 original_id）

import pandas as pd
import os

# ------------------- 配置 -------------------
ORIGINAL_TRAIN = "../../data/train.csv"                    # 原始训练集
PARAPHRASED_FILE = "train_paraphrased.csv"    # 你的改写文件
OUTPUT_FILE = "train_paraphrased_with_id.csv"  # 最终输出

print("步骤 1: 加载原始 train.csv，建立 true_q_i → original_id 映射...")
orig_df = pd.read_csv(ORIGINAL_TRAIN)

# 确保列名正确（根据你之前的代码）
# 如果列名是 true_q_i 和 original_id，直接用
if 'true_q_i' not in orig_df.columns or 'original_id' not in orig_df.columns:
    print("错误：原始 train.csv 必须包含 'true_q_i' 和 'original_id' 列")
    print("当前列名:", list(orig_df.columns))
    exit()

# 建立映射：true_q_i（标准化）→ original_id
# 标准化：去空格 + 小写，防止轻微格式差异
orig_df['q_norm'] = orig_df['true_q_i'].str.strip().str.lower()
mapping = dict(zip(orig_df['q_norm'], orig_df['original_id']))

print(f"映射建立完成：{len(mapping):,} 个唯一 true_q_i → original_id")

# ------------------- 加载改写数据 -------------------
print(f"\n步骤 2: 加载改写文件 {PARAPHRASED_FILE} ...")
para_df = pd.read_csv(PARAPHRASED_FILE)
print(f"改写数据行数: {len(para_df):,}")

# 标准化 true_q_i 用于匹配
para_df['q_norm'] = para_df['true_q_i'].str.strip().str.lower()

# 匹配并添加 Question_ID
print("正在匹配并添加 Question_ID ...")
para_df['Question_ID'] = para_df['q_norm'].map(mapping)

# 检查匹配率
matched = para_df['Question_ID'].notna().sum()
print(f"成功匹配: {matched:,}/{len(para_df):,} 条 ({matched/len(para_df):.2%})")

# 未匹配的打印前10个看看原因（调试用）
if matched < len(para_df):
    print("\n未匹配的 true_q_i 示例（前10个）：")
    print(para_df[para_df['Question_ID'].isna()]['true_q_i'].unique()[:10])

# 删除辅助列
para_df = para_df.drop(columns=['q_norm'])

# 重新排列列（方便查看）
cols = ['Question_ID', 'user_query', 'true_q_i', 'true_a_i', 
        'paraphrase_rank', 'dataset', 'original_id'] + \
       [c for c in para_df.columns if c not in ['Question_ID', 'user_query', 'true_q_i', 'true_a_i', 
                                                 'paraphrase_rank', 'dataset', 'original_id']]
para_df = para_df[cols]

# 保存
para_df.to_csv(OUTPUT_FILE, index=False)
print(f"\n完成！已保存带 Question_ID 的数据 → {OUTPUT_FILE}")

# 输出统计
print(f"\n最终统计：")
print(f"   唯一原始问题数 (Question_ID): {para_df['Question_ID'].nunique():,}")
print(f"   每题平均改写数: {len(para_df)/para_df['Question_ID'].nunique():.2f}")

步骤 1: 加载原始 train.csv，建立 true_q_i → original_id 映射...
映射建立完成：8,500 个唯一 true_q_i → original_id

步骤 2: 加载改写文件 train_paraphrased.csv ...
改写数据行数: 47,964
正在匹配并添加 Question_ID ...
成功匹配: 47,964/47,964 条 (100.00%)

完成！已保存带 Question_ID 的数据 → train_paraphrased_with_id.csv

最终统计：
   唯一原始问题数 (Question_ID): 7,687
   每题平均改写数: 6.24


In [17]:
# stats_train_paraphrased_with_id.py
# 统计 train_paraphrased_with_id.csv 的关键指标（用于论文）

import pandas as pd

FILE_PATH = "train_paraphrased_with_id.csv"

print("正在加载 train_paraphrased_with_id.csv ...")
df = pd.read_csv(FILE_PATH)

print(f"总行数: {len(df):,}")
print(f"总列数: {len(df.columns)}，列名: {list(df.columns)}\n")

# ------------------- 核心统计 -------------------
print("="*90)
print("RAL2M 训练集最终统计（可直接用于论文 Table 1）")
print("="*90)

# 1. 按 dataset 统计
stats = []

for dataset in sorted(df['dataset'].unique()):
    sub = df[df['dataset'] == dataset]
    
    unique_questions = sub['Question_ID'].nunique()           # 独立知识点
    total_rewrites = len(sub)                                 # 改写后总条数
    avg_rewrites = total_rewrites / unique_questions          # 平均改写数
    
    stats.append({
        'Dataset': dataset,
        'Domain': {
            'covidqa':'Biomed', 'pubmedqa':'Biomed', 'hotpotqa':'Wikipedia',
            'msmarco':'Web Search', 'finqa':'Finance', 'tatqa':'Finance',
            'cuad':'Legal', 'delucionqa':'General', 'emanual':'Manual',
            'expertqa':'Expert', 'techqa':'Tech', 'hagrid':'Finance'
        }.get(dataset, dataset),
        '#Unique QA': unique_questions,
        '#Rewrites': total_rewrites,
        'Avg Rewrites': f"{avg_rewrites:.2f}"
    })

stats_df = pd.DataFrame(stats)

# 添加总计行
total_unique = df['Question_ID'].nunique()
total_rewrites = len(df)
avg_total = total_rewrites / total_unique

total_row = {
    'Dataset': '**Total**',
    'Domain': '',
    '#Unique QA': total_unique,
    '#Rewrites': total_rewrites,
    'Avg Rewrites': f"{avg_total:.2f}"
}
stats_df = pd.concat([stats_df, pd.DataFrame([total_row])], ignore_index=True)

# 美化输出
print(stats_df.to_string(index=False, justify='center'))

print("="*90)

# 额外信息
print(f"\n其他关键统计：")
print(f"   唯一原始问题数 (Question_ID) : {total_unique:,}")
print(f"   改写后总训练样本数         : {total_rewrites:,}")
print(f"   平均每题改写数             : {avg_total:.2f}")
print(f"   数据增强倍率               : {avg_total:.1f}x")

# 保存统计表（用于论文）
stats_df.to_csv("train_paraphrased_statistics.csv", index=False)
print(f"\n统计表已保存 → train_paraphrased_statistics.csv")

正在加载 train_paraphrased_with_id.csv ...
总行数: 47,964
总列数: 10，列名: ['Question_ID', 'user_query', 'true_q_i', 'true_a_i', 'paraphrase_rank', 'dataset', 'original_id', 'dialogue_context', 'retrieved_docs', 'split']

RAL2M 训练集最终统计（可直接用于论文 Table 1）
 Dataset    Domain    #Unique QA  #Rewrites Avg Rewrites
  covidqa     Biomed     1468       8820        6.01    
 expertqa     Expert     1824      10944        6.00    
   hagrid    Finance      634       3828        6.04    
 hotpotqa  Wikipedia     2051      12306        6.00    
  msmarco Web Search     2011      12066        6.00    
**Total**                7687      47964        6.24    

其他关键统计：
   唯一原始问题数 (Question_ID) : 7,687
   改写后总训练样本数         : 47,964
   平均每题改写数             : 6.24
   数据增强倍率               : 6.2x

统计表已保存 → data_new/train_paraphrased_statistics.csv


In [24]:
# add_question_id_simple.py
# 完全不删任何数据！只从原始 train.csv 匹配 true_q_i → original_id → 添加 Question_ID

import pandas as pd

# ------------------- 配置 -------------------
ORIGINAL_TRAIN = "../../data/train.csv"           # 原始训练集
PARAPHRASED_FILE = "train_paraphrased.csv"       # 你的改写文件
OUTPUT_FILE = "train_paraphrased_with_id.csv"    # 输出（原样 + Question_ID）

print("加载原始 train.csv 建立 true_q_i → original_id 映射...")
orig_df = pd.read_csv(ORIGINAL_TRAIN)

# 标准化匹配键（去空格 + 小写）
orig_df['true_q_i_norm'] = orig_df['true_q_i'].str.strip().str.lower()
mapping = dict(zip(orig_df['true_q_i_norm'], orig_df['original_id']))

print(f"映射建立完成：{len(mapping):,} 个 true_q_i → original_id")

# ------------------- 加载改写数据 -------------------
print(f"\n加载改写文件: {PARAPHRASED_FILE}")
df = pd.read_csv(PARAPHRASED_FILE)
print(f"改写数据行数: {len(df):,}")

# ------------------- 添加 Question_ID -------------------
df['true_q_i_norm'] = df['true_q_i'].str.strip().str.lower()
df['Question_ID'] = df['true_q_i_norm'].map(mapping)

# 检查匹配率
matched = df['Question_ID'].notna().sum()
print(f"成功匹配 Question_ID: {matched:,}/{len(df):,} ({matched/len(df):.2%})")

# ------------------- 保存（原数据不动，只加一列） -------------------
# 删除辅助列，保留原始所有列 + 新增 Question_ID
df = df.drop(columns=['true_q_i_norm'], errors='ignore')

# 调整列顺序（Question_ID 放前面，更清晰）
cols = ['Question_ID'] + [c for c in df.columns if c != 'Question_ID']
df = df[cols]

df.to_csv(OUTPUT_FILE, index=False)
print(f"\n完成！已保存（零删除）→ {OUTPUT_FILE}")

# ------------------- 统计：每个 dataset 的唯一 QA 数量 -------------------
print("\n" + "="*80)
print("按 dataset 统计：唯一知识点数量（Question_ID）")
print("="*80)

stats = []
for dataset in sorted(df['dataset'].unique()):
    sub = df[df['dataset'] == dataset]
    unique_qa = sub['Question_ID'].nunique()
    total_rows = len(sub)
    avg_rewrites = total_rows / unique_qa if unique_qa > 0 else 0
    
    stats.append({
        'Dataset': dataset,
        '#Unique QA (Question_ID)': unique_qa,
        '#Total Rewrites': total_rows,
        'Avg Rewrites per QA': f"{avg_rewrites:.2f}"
    })

stats_df = pd.DataFrame(stats)

# 添加总计
total_unique = df['Question_ID'].nunique()
total_rows = len(df)
stats_df.loc[len(stats_df)] = {
    'Dataset': '**Total**',
    '#Unique QA (Question_ID)': total_unique,
    '#Total Rewrites': total_rows,
    'Avg Rewrites per QA': f"{total_rows/total_unique:.2f}"
}

print(stats_df.to_string(index=False))
print("="*80)

# 保存统计表
stats_df.to_csv("train_paraphrased_statistics.csv", index=False)
print(f"\n统计表已保存 → train_paraphrased_statistics.csv")

加载原始 train.csv 建立 true_q_i → original_id 映射...
映射建立完成：8,500 个 true_q_i → original_id

加载改写文件: train_paraphrased.csv
改写数据行数: 47,964
成功匹配 Question_ID: 47,964/47,964 (100.00%)

完成！已保存（零删除）→ train_paraphrased_with_id.csv

按 dataset 统计：唯一知识点数量（Question_ID）
  Dataset  #Unique QA (Question_ID)  #Total Rewrites Avg Rewrites per QA
  covidqa                      1468             8820                6.01
 expertqa                      1824            10944                6.00
   hagrid                       634             3828                6.04
 hotpotqa                      2051            12306                6.00
  msmarco                      2011            12066                6.00
**Total**                      7687            47964                6.24

统计表已保存 → train_paraphrased_statistics.csv


In [38]:
orig_df = pd.read_csv("train_paraphrased_final.csv")

orig_df["original_id"].nunique()


# ------------------- 统计：每个 dataset 的唯一 QA 数量 -------------------
print("\n" + "="*80)
print("按 dataset 统计：唯一知识点数量（Question_ID）")
print("="*80)

stats = []
for dataset in sorted(orig_df['dataset'].unique()):
    sub = orig_df[orig_df['dataset'] == dataset]
    unique_qa = sub['original_id'].nunique()
    total_rows = len(sub)
    avg_rewrites = total_rows / unique_qa if unique_qa > 0 else 0
    
    stats.append({
        'Dataset': dataset,
        '#Unique QA (Question_ID)': unique_qa,
        '#Total Rewrites': total_rows,
        'Avg Rewrites per QA': f"{avg_rewrites:.2f}"
    })

stats_df = pd.DataFrame(stats)

# 添加总计
total_unique = orig_df['original_id'].nunique()
total_rows = len(orig_df)
stats_df.loc[len(stats_df)] = {
    'Dataset': '**Total**',
    '#Unique QA (Question_ID)': total_unique,
    '#Total Rewrites': total_rows,
    'Avg Rewrites per QA': f"{total_rows/total_unique:.2f}"
}

print(stats_df.to_string(index=False))
print("="*80)


按 dataset 统计：唯一知识点数量（Question_ID）
  Dataset  #Unique QA (Question_ID)  #Total Rewrites Avg Rewrites per QA
  covidqa                      3989             3989                1.00
 expertqa                      5472             5472                1.00
   hagrid                      1902             1902                1.00
 hotpotqa                      6153             6153                1.00
  msmarco                      5545             5545                1.00
**Total**                     22982            23061                1.00


In [ ]:
orig_df

In [30]:
# find_missing_paraphrases.py
# 找出 train.csv 中哪些原始问题（original_id）在 train_paraphrased.csv 中完全没有出现

import pandas as pd

# ------------------- 配置 -------------------
ORIGINAL_TRAIN = "../../data/train.csv"           # 原始训练集
PARAPHRASED_FILE = "train_paraphrased.csv"   # 改写后的文件
MISSING_OUTPUT = "missing_from_paraphrased.csv"  # 保存缺失的原始问题

print("加载原始 train.csv ...")
orig_df = pd.read_csv(ORIGINAL_TRAIN)

print(f"原始训练样本数: {len(orig_df):,}")

print(f"\n加载改写文件: {PARAPHRASED_FILE}")
para_df = pd.read_csv(PARAPHRASED_FILE)

# 提取改写数据中的所有原始 ID（original_id 去掉 _p1/_p2/_p3 后缀）
para_df['original_id_base'] = para_df['original_id'].str.replace(r'_p[0-9]+$', '', regex=True)
paraphrased_ids = set(para_df['original_id_base'].unique())

print(f"改写文件中覆盖的原始问题数: {len(paraphrased_ids):,}")

# 原始训练集中的所有 original_id
original_ids = set(orig_df['original_id'].astype(str))

# 找出完全缺失的（一个改写都没有）
missing_ids = original_ids - paraphrased_ids

print(f"\n发现 {len(missing_ids):,} 个原始问题完全没有被改写！")

if len(missing_ids) == 0:
    print("恭喜！所有原始问题都至少有一条改写！")
else:
    # 保存缺失的原始样本
    missing_df = orig_df[orig_df['original_id'].astype(str).isin(missing_ids)].copy()
    
    # 可选：加一列说明原因
    missing_df['reason'] = "no_paraphrase_generated"
    
    missing_df.to_csv(MISSING_OUTPUT, index=False)
    print(f"已保存缺失样本 → {MISSING_OUTPUT}")
    print(f"   缺失样本数: {len(missing_df):,}")
    
    # 显示前 10 个看看
    print(f"\n前 10 个缺失样本示例：")
    print(missing_df[['original_id', 'user_query', 'true_q_i', 'dataset']].head(10).to_string(index=False))
    
    # 按 dataset 统计缺失情况
    print(f"\n缺失样本按 dataset 分布：")
    print(missing_df['dataset'].value_counts().to_string())

print(f"\n检查完成！")

加载原始 train.csv ...
原始训练样本数: 8,506

加载改写文件: train_paraphrased.csv
改写文件中覆盖的原始问题数: 7,692

发现 471 个原始问题完全没有被改写！
已保存缺失样本 → missing_from_paraphrased.csv
   缺失样本数: 471

前 10 个缺失样本示例：
             original_id                                                                                 user_query                                                                                   true_q_i  dataset
5ade82ba5542992fa25da7a3                                             Are Catasetum and Origanum in the same family?                                             Are Catasetum and Origanum in the same family? hotpotqa
5abcf5ec55429959677d6b76                                        Clone of clones played alongside a band from where?                                        Clone of clones played alongside a band from where? hotpotqa
5ab864945542990e739ec8e1 Substorm was described in qualitative terms by a scientist nominated for what seven times? Substorm was described in qualitative terms by a scientist n

In [33]:
# format_missing_like_paraphrased.py
# 把 missing_from_paraphrased.csv 格式化为和 train_paraphrased_with_id.csv 完全一致的列顺序

import pandas as pd

MISSING_FILE = "missing_from_paraphrased.csv"
OUTPUT_FILE  = "missing_formatted.csv"

print(f"加载缺失文件: {MISSING_FILE}")
df = pd.read_csv(MISSING_FILE)

print(f"原始行数: {len(df):,}")
print(f"原始列: {list(df.columns)}")

# 关键：从原始 train.csv 匹配 true_q_i → original_id → 生成 Question_ID
print("\n从原始 train.csv 匹配 Question_ID ...")
orig_train = pd.read_csv("../../data/train.csv")

# 标准化匹配
orig_train['true_q_i_norm'] = orig_train['true_q_i'].str.strip().str.lower()
df['true_q_i_norm'] = df['true_q_i'].str.strip().str.lower()

mapping = dict(zip(orig_train['true_q_i_norm'], orig_train['original_id']))
df['Question_ID'] = df['true_q_i_norm'].map(mapping)

# 检查是否全部匹配成功（应该 100%）
missing_match = df['Question_ID'].isna().sum()
if missing_match > 0:
    print(f"警告：有 {missing_match} 条未匹配到 Question_ID")
else:
    print("所有缺失样本成功匹配到 Question_ID")

# 统一列名（确保和 train_paraphrased_with_id.csv 完全一致）
df = df.rename(columns={
    'user_query': 'user_query',
    'true_q_i': 'true_q_i',
    'true_a_i': 'true_a_i',
    'dataset': 'dataset',
    'original_id': 'original_id'
})
# 添加缺失的列（填空值）
df['paraphrase_rank'] = 0        # 表示未改写
df['original_query'] = df['user_query']  # 原句即自己

# 最终列顺序（和你 train_paraphrased_with_id.csv 完全一致）
final_cols = [
    'Question_ID', 'user_query', 'true_q_i', 'true_a_i',
    'paraphrase_rank', 'dataset', 'original_id', 'original_query'
]

# 补全可能缺失的其他列（比如 dialogue_context, retrieved_docs 等）
existing_cols = df.columns.tolist()
for col in ['dialogue_context', 'retrieved_docs', 'split']:
    if col not in df.columns:
        df[col] = ""

# 最终列顺序（兼容所有可能的列）
final_cols = ['Question_ID', 'user_query', 'true_q_i', 'true_a_i',
              'paraphrase_rank', 'dataset', 'original_id', 'original_query'] + \
             [c for c in df.columns if c not in final_cols]

df = df[final_cols]

df.to_csv(OUTPUT_FILE, index=False)
print(f"\n格式化完成！已保存 → {OUTPUT_FILE}")
print(f"   行数: {len(df):,}")
print(f"   列顺序已和 train_paraphrased_with_id.csv 完全一致")

print(f"\n前 3 行预览：")
print(df.head(3)[final_cols].to_string(index=False))

加载缺失文件: missing_from_paraphrased.csv
原始行数: 471
原始列: ['user_query', 'dialogue_context', 'retrieved_docs', 'true_q_i', 'true_a_i', 'dataset', 'split', 'original_id', 'reason']

从原始 train.csv 匹配 Question_ID ...
所有缺失样本成功匹配到 Question_ID

格式化完成！已保存 → missing_formatted.csv
   行数: 471
   列顺序已和 train_paraphrased_with_id.csv 完全一致

前 3 行预览：
             Question_ID                                                                                 user_query                                                                                   true_q_i                                                                                                                                  true_a_i  paraphrase_rank  dataset              original_id                                                                             original_query  dialogue_context                                                                                                                                                                    

In [36]:
# finalize_train_paraphrased.py
# 每个 Question_ID 只保留 3 条语义最不同的改写 → 生成最终训练集

import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch

INPUT_FILE = "train_paraphrased_with_id.csv"
OUTPUT_FILE = "train_paraphrased_final.csv"

print("加载数据...")
df = pd.read_csv(INPUT_FILE)

print(f"原始改写条数: {len(df):,}")
print(f"原始 Question_ID 数量: {df['Question_ID'].nunique():,}")

# 加载模型（只加载一次）
print("\n加载 all-MiniLM-L6-v2 用于多样性选择...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

final_rows = []

print("\n开始处理每个 Question_ID，只保留语义最不同的 3 条改写...")
for qid, group in df.groupby('Question_ID'):
    if len(group) <= 3:
        final_rows.extend(group.to_dict('records'))
        continue

    # 提取所有改写问题
    queries = group['user_query'].tolist()
    
    # 计算嵌入
    embs = model.encode(queries, convert_to_tensor=True, normalize_embeddings=True)
    
    # 计算两两余弦相似度
    sim_matrix = util.cos_sim(embs, embs)
    
    # 方法：从最不相似的开始选（贪心）
    selected_idx = []
    remaining = list(range(len(queries)))
    
    # 选第一个（随意）
    first = 0
    selected_idx.append(first)
    remaining.remove(first)
    
    # 再选两个最不相似的
    for _ in range(2):
        if not remaining:
            break
        # 计算剩余的与已选的平均相似度（越小越好）
        sim_to_selected = sim_matrix[remaining][:, selected_idx].mean(dim=1)
        best_idx = remaining[sim_to_selected.argmin().item()]
        selected_idx.append(best_idx)
        remaining.remove(best_idx)
    
    # 保存选中的 3 条
    selected_rows = group.iloc[selected_idx].copy()
    selected_rows['paraphrase_rank'] = range(1, 4)  # 重新编号 1,2,3
    final_rows.extend(selected_rows.to_dict('records'))

# 转为 DataFrame
final_df = pd.DataFrame(final_rows)

# 最终列顺序（美观）
final_cols = [
    'Question_ID', 'user_query', 'true_q_i', 'true_a_i',
    'paraphrase_rank', 'dataset', 'original_id'
] + [c for c in final_df.columns if c not in final_cols]

final_df = final_df[final_cols]

# 保存
final_df.to_csv(OUTPUT_FILE, index=False)

print(f"\n处理完成！")
print(f"   最终训练样本数: {len(final_df):,}")
print(f"   最终 Question_ID 数: {final_df['Question_ID'].nunique():,}")
print(f"   每题平均改写数: {len(final_df)/final_df['Question_ID'].nunique():.2f}")
print(f"   已保存 → {OUTPUT_FILE}")

# 统计
print(f"\n每题正好 3 条改写的 Question_ID 数量: {final_df['Question_ID'].value_counts().eq(3).sum():,}")
print("完美！你的最终训练集已准备就绪，可直接用于 RAL2M 训练！")

加载数据...
原始改写条数: 47,964
原始 Question_ID 数量: 7,687

加载 all-MiniLM-L6-v2 用于多样性选择...



开始处理每个 Question_ID，只保留语义最不同的 3 条改写...

处理完成！
   最终训练样本数: 23,061
   最终 Question_ID 数: 7,687
   每题平均改写数: 3.00
   已保存 → train_paraphrased_final.csv

每题正好 3 条改写的 Question_ID 数量: 7,687
完美！你的最终训练集已准备就绪，可直接用于 RAL2M 训练！
